In [ ]:
# Google Colab setup
!pip install -q polars plotly gdown python-dotenv

# Mount Google Drive for datasets
from google.colab import drive
drive.mount("/content/drive")


# ════════════════════════════════════════════════════════════════════════
# ASCENT01 — Exercise 2: Filtering and Transforming Data
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Master Boolean logic and method chaining in Polars — the
#   core pattern for slicing and reshaping data without writing loops.
#
# TASKS:
#   1. Filter HDB resale transactions by town, price, and date
#   2. Select and rename columns to focus your analysis
#   3. Create new derived columns with with_columns()
#   4. Sort results and combine filters with & and |
#   5. Chain operations together to build a clean analysis pipeline
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import polars as pl

from shared import ASCENTDataLoader



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
hdb = loader.load("ascent01", "hdb_resale.parquet")

print("=== HDB Resale Dataset ===")
print(f"Shape: {hdb.shape}")
print(f"Columns: {hdb.columns}")
print(hdb.head(3))



## TASK 1: Boolean filters — keep only rows that match a condition


In [ ]:
# A Boolean is True or False. Polars lets you write conditions using
# pl.col("column_name") and comparison operators: ==, !=, >, <, >=, <=

# Filter for a single town
# pl.col("town") creates a column reference — like pointing at the column
# TODO: Filter hdb to rows where town equals "ANG MO KIO"
ang_mo_kio = hdb.filter(pl.col("town") == ____)
print(f"\nAng Mo Kio transactions: {ang_mo_kio.height:,}")

# Filter by price range
# Note: use & for AND (both must be True), | for OR (either can be True)
# You MUST wrap each condition in parentheses when combining with & or |
# TODO: Complete the filter — keep rows where resale_price is between 300k and 500k
affordable = hdb.filter(
    (pl.col("resale_price") >= 300_000) & (pl.col("resale_price") <= ____)
)
print(f"Transactions S$300k–500k: {affordable.height:,}")

# Filter by flat type
# TODO: Filter hdb to rows where flat_type equals "4 ROOM"
four_room = hdb.filter(pl.col("flat_type") == ____)
print(f"4-room flats: {four_room.height:,}")

# Combine multiple conditions — Ang Mo Kio 4-room under S$500k
# TODO: Combine three conditions with & — town, flat_type, and price <= 500k
amk_4room_affordable = hdb.filter(
    (pl.col("town") == "ANG MO KIO")
    & (pl.col("flat_type") == "4 ROOM")
    & (pl.col("resale_price") <= ____)
)
print(f"AMK 4-room under S$500k: {amk_4room_affordable.height:,}")

# .is_in() filters for any of several values — cleaner than chaining ==
central_towns = ["BISHAN", "TOA PAYOH", "QUEENSTOWN", "BUKIT MERAH"]
# TODO: Use .is_in() to filter hdb rows where town is in central_towns
central = hdb.filter(pl.col("town").is_in(____))
print(f"Central towns transactions: {central.height:,}")



## TASK 2: Select and rename columns


In [ ]:
# .select() keeps only the columns you name — everything else is dropped.
# This is important: always work with the smallest DataFrame that answers
# your question. Extra columns waste memory and clutter your output.

core_cols = hdb.select(
    "month",
    "town",
    "flat_type",
    "floor_area_sqm",
    "resale_price",
)
print(f"\nAfter select: {core_cols.columns}")

# .rename() changes column names — useful when names are awkward or long
# Pass a dict: {"old_name": "new_name"}
# TODO: Rename "month" to "sale_month" in the rename dict
renamed = core_cols.rename(
    {
        ____: "sale_month",
        "floor_area_sqm": "area_sqm",
        "resale_price": "price",
    }
)
print(f"After rename: {renamed.columns}")
print(renamed.head(3))



## TASK 3: Derived columns with with_columns()


In [ ]:
# .with_columns() adds new columns (or replaces existing ones) without
# removing any other columns. This is how you engineer features.
# .alias() gives the new column a name.

hdb = hdb.with_columns(
    # Price per square metre — a normalised measure for fair comparison
    # TODO: Divide resale_price by floor_area_sqm, alias as "price_per_sqm"
    (pl.col("resale_price") / pl.col("floor_area_sqm")).alias(____)
)

# You can add multiple columns in one call — more efficient than chaining
hdb = hdb.with_columns(
    # Parse the "month" string (e.g. "2023-01") into a proper date
    # str.to_date() converts text → date; "%Y-%m" is the format pattern
    # TODO: Parse "month" with format "%Y-%m", alias as "transaction_date"
    pl.col("month").str.to_date(____).alias("transaction_date"),
    # Extract the year as an integer for grouping later
    # TODO: Slice the first 4 characters of "month" and cast to Int32, alias "year"
    pl.col("month").str.slice(0, ____).cast(pl.Int32).alias("year"),
)

print(f"\n=== After adding derived columns ===")
print(hdb.select("month", "transaction_date", "year", "price_per_sqm").head(5))



## TASK 4: Conditional columns with pl.when().then().otherwise()


In [ ]:
# pl.when() is Polars' if/else for column creation.
# Think of it as: "When this condition, give this value; otherwise, that value."
# You can chain .when().then() as many times as you need.

hdb = hdb.with_columns(
    pl.when(pl.col("resale_price") < 350_000)
    .then(pl.lit("budget"))
    .when(pl.col("resale_price") < 500_000)
    .then(pl.lit("mid_range"))
    # TODO: Add a .when().then() branch for "premium" (price < 700_000)
    .when(pl.col("resale_price") < ____)
    .then(pl.lit("premium"))
    .otherwise(pl.lit("luxury"))
    .alias("price_tier")
)

# pl.lit() wraps a constant value into a Polars expression
# Without it, Polars would treat the string as a column name

# Count each tier
tier_counts = (
    hdb.group_by("price_tier")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
print(f"\n=== Price Tier Distribution ===")
print(tier_counts)



## TASK 5: Sort and chain operations — building a pipeline


In [ ]:
# .sort() orders rows. descending=True puts the largest values first.
# sort() on multiple columns: first column is the primary sort key.

# Method chaining: instead of saving each intermediate step to a variable,
# you attach the next operation with a dot. Polars is designed for this.
# Reading left-to-right (or top-to-bottom) tells the story of your analysis.

recent_premium = (
    hdb
    # Step 1: Keep only recent years
    # TODO: Filter to rows where year is 2020 or later
    .filter(pl.col("year") >= ____)
    # Step 2: Keep only premium and luxury flats
    # TODO: Use is_in() to keep rows where price_tier is "premium" OR "luxury"
    .filter(pl.col("price_tier").is_in(____))
    # Step 3: Keep only the columns we care about
    .select(
        "transaction_date",
        "town",
        "flat_type",
        "price_per_sqm",
        "price_tier",
        "resale_price",
    )
    # Step 4: Sort by price descending to see the most expensive first
    # TODO: Sort by "resale_price" with descending=True
    .sort("resale_price", descending=____)
)

print(f"\n=== Recent Premium/Luxury Transactions (2020+) ===")
print(f"Count: {recent_premium.height:,}")
print(recent_premium.head(10))

# The town with the most high-value transactions
top_towns = (
    recent_premium.group_by("town")
    .agg(
        pl.len().alias("transaction_count"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("price_per_sqm").median().alias("median_price_sqm"),
    )
    .sort("transaction_count", descending=True)
)

print(f"\n=== Towns with Most Premium/Luxury Transactions (2020+) ===")
print(top_towns.head(10))

# Polars chaining reads like a sentence:
# "Take the HDB data, filter to recent years, filter to premium tier,
#  select relevant columns, sort by price" — each step is clear.

print("\n✓ Exercise 2 complete — filtering, transforming, and method chaining")

